# Smoothed daily-rainfall animation

This notebook builds an animated map of consensus daily rainfall over the UK,
**smoothed by interpolating between consecutive days**. Instead of one abrupt
frame per day, several intermediate frames are drawn between each day and the
next (linear interpolation per station, missing values treated as zero), so the
rainfall field appears to evolve continuously.

Each frame is the same map produced by `plot_daily_rainfall_map.py` (median over
the 5 ensemble members, only located stations for the matching `matched_year`),
so the animation shares the static/interactive maps' styling: tall UK framing,
`YlGnBu` square-root colour scale, values in **inches**, coastlines and borders.

Rendering a full multi-year run is thousands of frames, so the work is sharded
and run locally with the staged animation pipeline, and the final MP4 is then
displayed here.

## How the pipeline works

The animation is produced by four local stages run in order:

| Stage | Script | Local runner | Purpose |
|-------|--------|--------------|---------|
| precompute | `scripts/slurm/render_precompute.sbatch` | `scripts/local/submit_local.sh` | write `manifest.json` describing every frame and shard boundaries |
| render | `scripts/slurm/render_array.sbatch` | `scripts/local/run_array_local.sh` | render each shard's contiguous frame range in parallel |
| validate | `scripts/slurm/render_validate.sbatch` | `scripts/local/submit_local.sh` | confirm every expected `frame_NNNNNNN.png` exists |
| encode | `scripts/slurm/render_encode.sbatch` | `scripts/local/submit_local.sh` | `ffmpeg` the frame sequence into an H.264 MP4 |

**Deterministic frame indexing.** Every frame has a global index `0 .. total-1`
fixed by date range and `frames_per_day`. Each render shard owns a contiguous
index block, so shards never collide and one failed shard can be rerun cleanly.

**Interpolation density.** `RENDER_FRAMES_PER_DAY` is the number of frames
covering each day->next-day step (default `6` means source day plus 5
interpolated in-between frames). Raise it for a smoother, longer video.

**Where things are stored** (under `$PDIR/animation/`):

- `manifest.json` - run description read by every stage
- `frames/frame_NNNNNNN.png` - rendered frames
- `rainfall_<start>_<end>.mp4` (+ `.mp4.json`) - final video

Everything (date range, interpolation density, shard count, fps, colour scale
and stage resources) is configured in `scripts/slurm/config.sh` via `RENDER_*`,
and can be overridden at submit time as shown below.

In [ ]:
# Setup: locate the repository root and local submit helper.

import os
import subprocess
from pathlib import Path

import src.rainfall_rescue_sqlite as _pkg

repo_root = Path(_pkg.__file__).resolve().parents[2]
local_submit_script = repo_root / "scripts" / "local" / "submit_local.sh"

pdir = Path(os.environ["PDIR"])
animation_dir = pdir / "animation"

print(f"repo_root:         {repo_root}")
print(f"local submit:      {local_submit_script}")
print(f"animation dir:     {animation_dir}")

repo_root:      /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-QC-MO
submit script:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-QC-MO/scripts/slurm/submit_animation.sh
animation dir:  /data/scratch/philip.brohan/ADRQ/animation


## Submit a test run (a single year)

Start with one year (1931) to check the whole pipeline end-to-end before
committing time to the full record. The environment variables below override
defaults in `config.sh` for this run only.

- `RENDER_DATE_START` / `RENDER_DATE_END` - calendar range to animate
- `RENDER_FRAMES_PER_DAY` - interpolation density
- `RENDER_NUM_SHARDS` - number of render shards
- `RENDER_FPS` - playback frame rate of the MP4

The cell runs the local `animation` pipeline and prints stage output.

In [ ]:
# Submit the test-year animation pipeline locally.

test_env = {
    **os.environ,
    "RENDER_DATE_START": "1931-01-01",
    "RENDER_DATE_END": "1931-12-31",
    "RENDER_FRAMES_PER_DAY": "3",   # source day + 2 interpolated frames
    "RENDER_NUM_SHARDS": "50",
    "RENDER_FPS": "30",
}

result = subprocess.run(
    ["bash", str(local_submit_script), "animation"],
    env=test_env,
    cwd=str(repo_root),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(f"local animation pipeline failed with code {result.returncode}")

Submitted precompute job: 28540935
Submitted render array: 28540936 (0-49)
Submitted validate job: 28540937
Submitted encode job: 28540938

Animation pipeline submitted. Track with:  squeue -u $USER
Final video lands in: /data/scratch/philip.brohan/ADRQ/animation/ (see manifest output_path)



## Monitor the run

Re-run the cell below to watch progress. The pipeline is done when the final
MP4 appears in `$PDIR/animation/`. Stage logs are written to `$PDIR/local_logs/`.

In [ ]:
# Show recent local animation logs and any finished videos.

log_dir = pdir / "local_logs"
print("Recent local logs:")
if log_dir.exists():
    logs = sorted(log_dir.glob("*render*.log"), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in logs[:12]:
        print(f"  {p.name}")
    if not logs:
        print("  (no render logs yet)")
else:
    print("  (local log directory does not exist yet)")

print("\nRendered videos in", animation_dir, ":")
if animation_dir.exists():
    videos = sorted(animation_dir.glob("*.mp4"))
    for v in videos:
        print(f"  {v.name}  ({v.stat().st_size / 1e6:.1f} MB)")
    if not videos:
        n_frames = len(list((animation_dir / "frames").glob("*.png"))) if (animation_dir / "frames").exists() else 0
        print(f"  (no MP4 yet; {n_frames} frames rendered so far)")
else:
    print("  (animation directory does not exist yet)")

             JOBID                 NAME    STATE       TIME  NODES NODELIST(REASON)
            592443                 find  PENDING       0:00      1 (BeginTime)


Rendered videos in /data/scratch/philip.brohan/ADRQ/animation :
  rainfall_1860-01-01_1960-12-31.mp4  (628.0 MB)
  rainfall_1900-01-01_1939-12-31.mp4  (148.2 MB)
  rainfall_1930-01-01_1931-12-31.mp4  (10.1 MB)
  rainfall_1931-01-01_1931-12-31.mp4  (8.4 MB)


## Display the finished animation

Once the encode stage has completed, the cell below embeds the MP4 inline. It
picks the most recently modified video in the animation directory.

In [ ]:
# Embed the most recent rendered animation.

from IPython.display import Video, display

videos = sorted(animation_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
if not videos:
    raise FileNotFoundError(
        f"No MP4 found in {animation_dir}. Has the encode stage finished?"
    )

latest_video = videos[-1]
print(f"Displaying: {latest_video}")
display(Video(str(latest_video), embed=True, width=500))

## Run the full record

When the test year looks right, animate the full available date range. This is a
much larger job — raise `RENDER_NUM_SHARDS` so the frames render in parallel
across many array tasks. Set `RENDER_DATE_START` / `RENDER_DATE_END` to the span
you want (the whole matched record, or any sub-period).

The cell is not executed by default — change `submit_full = True` to launch it.

In [ ]:
# Submit the full-range animation pipeline (guarded so it does not run by accident).

submit_full = True  # set to True to launch the full-record render

full_env = {
    **os.environ,
    "RENDER_DATE_START": "1860-01-01",
    "RENDER_DATE_END": "1960-12-31",
    "RENDER_FRAMES_PER_DAY": "3",
    "RENDER_NUM_SHARDS": "500",
    "RENDER_FPS": "30",
}

if submit_full:
    result = subprocess.run(
        ["bash", str(local_submit_script), "animation"],
        env=full_env,
        cwd=str(repo_root),
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise SystemExit(f"local animation pipeline failed with code {result.returncode}")
else:
    print("submit_full is False - not submitting. Set it to True to launch the full render.")

Submitted precompute job: 28548968
Submitted render array: 28548969 (0-499)
Submitted validate job: 28548970
Submitted encode job: 28548971

Animation pipeline submitted. Track with:  squeue -u $USER
Final video lands in: /data/scratch/philip.brohan/ADRQ/animation/ (see manifest output_path)

